In [ ]:
from pathlib import Path
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

DATA_DIR = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/EXPERIMENTS/exp1a_train_M_detect_F")

LATENT_DIM = 16         
WIDTH_FIRST = 256       
BETA_KL = 0.01          
BATCH_SIZE = 32         
DROPOUT = 0.2          
LEARNING_RATE = 1e-3   
EPOCHS = 1000           
PATIENCE = 30


ANGLE_CHANNELS = [
    "LFootProgressAngles", "RFootProgressAngles",
    "LHipAngles",          "RHipAngles",
    "LKneeAngles",         "RKneeAngles",
    "LAnkleAngles",        "RAnkleAngles",
    "LForeFootAngles",     "RForeFootAngles",
    "LSpineAngles",        "RSpineAngles",
    "LShoulderAngles",     "RShoulderAngles",
    "LElbowAngles",        "RElbowAngles",
    "LWristAngles",        "RWristAngles",
    "LNeckAngles",         "RNeckAngles",
    "LPelvisAngles",       "RPelvisAngles",
    "LThoraxAngles",       "RThoraxAngles",
    "LHeadAngles",         "RHeadAngles",
]
NUM_JOINTS = len(ANGLE_CHANNELS)  

X_train_final  = np.load(DATA_DIR / "X_train.npy")          # (N, 100, 26, 3)
X_val          = np.load(DATA_DIR / "X_val.npy")
X_test_normal  = np.load(DATA_DIR / "X_test_normal.npy")
X_test_anomaly = np.load(DATA_DIR / "X_test_anomaly.npy")

subjects_train = np.load(DATA_DIR / "subjects_train.npy", allow_pickle=True)
subjects_val   = np.load(DATA_DIR / "subjects_val.npy", allow_pickle=True)
subjects_test_normal  = np.load(DATA_DIR / "subjects_test_normal.npy", allow_pickle=True)
subjects_test_anomaly = np.load(DATA_DIR / "subjects_test_anomaly.npy", allow_pickle=True)

print(f"  TRAIN (Mężczyźni S1, 80%):   {X_train_final.shape} - {len(np.unique(subjects_train))} osób")
print(f"  VAL   (Mężczyźni S1, 20%):   {X_val.shape} - {len(np.unique(subjects_val))} osób")
print(f"  TEST Normal (Mężczyźni S2):  {X_test_normal.shape} - {len(np.unique(subjects_test_normal))} osób")
print(f"  TEST Anomaly (Kobiety S1+S2): {X_test_anomaly.shape} - {len(np.unique(subjects_test_anomaly))} osób")

# Flatten: (N, 100, 26, 3)  (N, 7800)
X_train_flat        = X_train_final.reshape(X_train_final.shape[0], -1)
X_val_flat          = X_val.reshape(X_val.shape[0], -1)
X_test_normal_flat  = X_test_normal.reshape(X_test_normal.shape[0], -1)
X_test_anomaly_flat = X_test_anomaly.reshape(X_test_anomaly.shape[0], -1)

print(f"  Train:        {X_train_flat.shape}")
print(f"  Val:          {X_val_flat.shape}")
print(f"  Test Normal:  {X_test_normal_flat.shape}")
print(f"  Test Anomaly: {X_test_anomaly_flat.shape}")

# Z-score fit tylko na TRAIN
scaler = StandardScaler()
X_train_norm        = scaler.fit_transform(X_train_flat)
X_val_norm          = scaler.transform(X_val_flat)
X_test_normal_norm  = scaler.transform(X_test_normal_flat)
X_test_anomaly_norm = scaler.transform(X_test_anomaly_flat)

print(f"  Train:        mean={X_train_norm.mean():.3f}, std={X_train_norm.std():.3f}")
print(f"  Val:          mean={X_val_norm.mean():.3f}, std={X_val_norm.std():.3f}")
print(f"  Test Normal:  mean={X_test_normal_norm.mean():.3f}, std={X_test_normal_norm.std():.3f}")
print(f"  Test Anomaly: mean={X_test_anomaly_norm.mean():.3f}, std={X_test_anomaly_norm.std():.3f}")

input_dim = X_train_norm.shape[1]

In [ ]:
# SETUP

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

In [ ]:
# Create a sampling layer

In [ ]:
class Sampling(layers.Layer):
    """Reparameterization trick: z = μ + ε·σ,  ε∼N(0,I)."""
    
    def call(self, inputs):
        z_mean, z_logvar = inputs
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_logvar) * epsilon

In [ ]:
# Build the encoder (

In [ ]:
encoder_inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(WIDTH_FIRST, activation="relu")(encoder_inputs)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(WIDTH_FIRST // 2, activation="relu")(x)
x = layers.Dropout(DROPOUT)(x)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

print("\nENCODER:")
encoder.summary()

In [ ]:
# Build the decoder

In [ ]:
latent_input = keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(WIDTH_FIRST // 2, activation="relu")(latent_input)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(WIDTH_FIRST, activation="relu")(x)
x = layers.Dropout(DROPOUT)(x)
decoder_output = layers.Dense(input_dim, activation="linear")(x)
decoder = keras.Model(latent_input, decoder_output, name="decoder")

print("\nDecoder:")
decoder.summary()

In [ ]:
# Define the VAE

In [ ]:
"""β-VAE: loss = MSE + β·KL."""
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=BETA_KL, **kw):
        super().__init__(**kw)
        self.encoder     = encoder
        self.decoder     = decoder
        self.beta        = beta
        self.loss_tracker= keras.metrics.Mean(name="loss")
        self.mse_tracker = keras.metrics.Mean(name="mse")
        self.kl_tracker  = keras.metrics.Mean(name="kl")

    @property
    def metrics(self):          # <– Keras będzie to resetował co epokę
        return [self.loss_tracker, self.mse_tracker, self.kl_tracker]

    def call(self, x):
        _, _, z = self.encoder(x)
        return self.decoder(z)

    def train_step(self, x):
        with tf.GradientTape() as tape:
            mu, logvar, z = self.encoder(x, training=True)
            logvar = tf.clip_by_value(logvar, -10.0, 10.0)       #BEZ TEGO
            recon = self.decoder(z, training=True)
            mse = tf.reduce_mean(tf.square(x - recon), axis=1)
            kl  = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
            #loss = tf.reduce_mean(mse + self.beta * kl)
            loss = tf.reduce_mean(mse + self.beta * kl)

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))

        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }

    # val step (żeby działał val_loss)
    def test_step(self, x):
        mu, logvar, z = self.encoder(x, training=False)
        logvar = tf.clip_by_value(logvar, -10.0, 10.0)
        recon  = self.decoder(z, training=False)
        mse    = tf.reduce_mean(tf.square(x - recon), axis=1)
        kl     = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
        loss   = tf.reduce_mean(mse + self.beta * kl)
        self.loss_tracker.update_state(loss)
        self.mse_tracker.update_state(tf.reduce_mean(mse))
        self.kl_tracker.update_state(tf.reduce_mean(kl))
        return {
            "loss": self.loss_tracker.result(),
            "mse":  self.mse_tracker.result(),
            "kl":   self.kl_tracker.result()
        }




In [ ]:
# Train the VAE

In [ ]:
vae = VAE(encoder, decoder, beta=BETA_KL)
vae.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE, clipnorm=1.0))

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

history = vae.fit(
    X_train_norm,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(X_val_norm, None),
    callbacks=[early_stop],
    verbose=1
)

# 6. Evaluation
def compute_mse(model, X):
    mu, logvar, z = model.encoder.predict(X, verbose=0)
    X_recon = model.decoder.predict(z, verbose=0)
    return np.mean((X - X_recon)**2, axis=1)

mse_train = compute_mse(vae, X_train_norm)                   # Train (M S1 80%)
mse_val = compute_mse(vae, X_val_norm)                       # Val (M S1 20%)
mse_test_normal = compute_mse(vae, X_test_normal_norm)       # Test Normal (M S2)
mse_test_anomaly = compute_mse(vae, X_test_anomaly_norm)     # Test Anomaly

y_train_val = np.concatenate([
    np.zeros(len(mse_train)),  # train = 0
    np.ones(len(mse_val))      # val = 1
])
mse_train_val = np.concatenate([mse_train, mse_val])


from sklearn.metrics import roc_curve, confusion_matrix

fpr_tv, tpr_tv, thresholds_tv = roc_curve(y_train_val, mse_train_val)

def f1_at_threshold(thr, y_true, y_score):
    y_pred = (y_score > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    
    return f1, prec, rec, tn, fp, fn, tp

y_true = np.concatenate([
    np.zeros(len(mse_test_normal)),    # 0 = normal (Mężczyźni S2)
    np.ones(len(mse_test_anomaly))     # 1 = anomaly (Kobiety)
])
y_score = np.concatenate([mse_test_normal, mse_test_anomaly])


y_pred = (y_score > threshold).astype(int)

from sklearn.metrics import roc_auc_score, average_precision_score

auc = roc_auc_score(y_true, y_score)
ap = average_precision_score(y_true, y_score)
separation = mse_test_anomaly.mean() / mse_test_normal.mean()


tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()


test_prec = tp / (tp + fp + 1e-9)
test_rec = tp / (tp + fn + 1e-9)
test_f1 = 2 * test_prec * test_rec / (test_prec + test_rec + 1e-9)


fpr, tpr, _ = roc_curve(y_true, y_score)


print(f"  AUC: {auc:.3f}")
print(f"  AP (Average Precision): {ap:.3f}")
print(f"  Separation (F/M): {separation:.3f}x")
print(f"  Threshold used: {threshold:.6f} (from train/val split)")


print(f"  True Negatives (M→M):  {tn:3d}/{len(mse_test_normal):3d} ({tn/len(mse_test_normal)*100:5.1f}%)")
print(f"  False Positives (M→F): {fp:3d}/{len(mse_test_normal):3d} ({fp/len(mse_test_normal)*100:5.1f}%)")
print(f"  False Negatives (F→M): {fn:3d}/{len(mse_test_anomaly):3d} ({fn/len(mse_test_anomaly)*100:5.1f}%)")
print(f"  True Positives (F→F):  {tp:3d}/{len(mse_test_anomaly):3d} ({tp/len(mse_test_anomaly)*100:5.1f}%)")


print(f"  TPR (Sensitivity/Recall): {test_rec:.3f}")
print(f"  TNR (Specificity):        {tn/(tn+fp):.3f}")
print(f"  FPR:                      {fp/(fp+tn):.3f}")
print(f"  FNR:                      {fn/(fn+tp):.3f}")
print(f"  Precision:                {test_prec:.3f}")
print(f"  F1-Score:                 {test_f1:.3f}")




fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history["mse"], label="Train MSE", linewidth=2, color='blue')
ax.plot(history.history["val_mse"], label="Val MSE", linewidth=2, color='cyan')
ax.plot(history.history["kl"], label="Train KL", linewidth=2, linestyle='--', color='orange')
ax.plot(history.history["val_kl"], label="Val KL", linewidth=2, linestyle='--', color='red')
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Training History", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Histogram 
plt.figure(figsize=(10, 5))

plt.hist(mse_test_normal, bins=40, alpha=0.7, 
         label=f'Normal (M S2, n={len(mse_test_normal)})', 
         color='blue', edgecolor='black')
plt.hist(mse_test_anomaly, bins=40, alpha=0.7, 
         label=f'Anomaly (F all, n={len(mse_test_anomaly)})', 
         color='red', edgecolor='black')
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, 
            label=f'Threshold')

plt.xlabel('Reconstruction Error (MSE)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Reconstruction Errors (TEST SET)', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

textstr = f'AUC = {auc:.3f}\nAP = {ap:.3f}\nSep = {separation:.2f}x'
plt.text(0.98, 0.97, textstr, transform=plt.gca().transAxes,
         fontsize=11, ha='right', va='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# ROC
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, color='blue', label=f'Dense VAE (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Sex Difference Detection (M→F)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
